# Wykład 11 — Od pojedynczego znaku do tekstu

Jak program zapisuje zdanie „Ala ma kota”, a potem odnajduje w nim słowo „kota”? W C napis to konwencja zapisu w pamięci. Czytamy kolejne elementy typu `char` i kończymy na pierwszym znaku o wartości zero.

W W2 poznaliśmy `char` i sposób przedstawiania danych, w W3 strumienie wejścia i wyjścia, a w W5 pętle. Z W6–W7 przydadzą się adresy, wskaźniki i tablice. Z plików i alokacji omawianych w W8–W10 skorzystamy, by wczytać tekst i przeszukać go linia po linii.

Po wykładzie umiesz przewidzieć zawartość tablicy znaków, dobrać pojemność bufora i uzasadnić, dlaczego odczyt lub zapis mieści się w jego granicach. Złożysz też prosty program wyszukujący fragment tekstu w pliku.

Każda wykonywalna komórka to osobny program C11. Przed uruchomieniem zapisz przewidywany wynik, porównaj go z wydrukiem i zmień jeden szczegół. Programy w kernelu C korzystają z gotowych danych. Warianty terminalowe czytają ze standardowego wejścia.

## W11.1 — Znak i jego wartość

Jak komputer odróżnia cyfrę na ekranie od liczby w obliczeniu?

W przykładach przyjmujemy Linux z GCC i kodowanie zgodne z ASCII. Znak drukowalny ma postać widoczną na ekranie, na przykład `A` albo `7`. Spacja zajmuje miejsce, a znaki sterujące, takie jak `\n` i `\t`, zmieniają układ wydruku. NUL zapisujemy jako `'\0'`. Ma wartość zero. Makro `NULL` oznacza stałą wskaźnika pustego.

`sizeof(char)` zawsze daje 1 bajt C. Liczbę bitów w bajcie określa `CHAR_BIT`. Zwykły `char` może na danej platformie mieć zakres typu `signed char` albo `unsigned char`. W UTF-8 litera spoza ASCII może zajmować kilka bajtów. Funkcje napisowe w tym wykładzie operują na bajtach.

**Przewidź:** które z wartości `'7'`, `7`, `'0'`, `0` i `'\0'` są równe? Jak zmieni się wydruk po `\n`?

In [ ]:
#include <limits.h>
#include <stdio.h>

int main(void)
{
    char cyfra = '7';

    printf("char: %zu bajt C, %d bitów w bajcie\n", sizeof(char), CHAR_BIT);
    printf("'%c' ma kod %d, liczba 7 ma wartość %d\n", cyfra, cyfra, 7);
    printf("'0': %d, 0: %d, '\\0': %d\n", '0', 0, '\0');
    printf("A\tB\nC\n");
    return 0;
}

**Interpretacja.** W ASCII `'7'` ma kod 55, a `'0'` ma kod 48. Wartości `0` i `'\0'` są równe. `%c` wypisał znak cyfry, a `%d` jego wartość liczbową. Tabulator oddzielił `A` od `B`, a `\n` przeniósł `C` do następnej linii.

**Spróbuj.** Przewidź wynik wyrażenia `cyfra - '0'`. Sprawdź go dla dwóch cyfr.

## W11.2 — Gdzie kończy się napis?

Skąd funkcja wypisująca tekst wie, kiedy przestać czytać tablicę?

Zbudujemy napis `Hi` ręcznie, element po elemencie. `%c` pobiera jeden znak. `%s` zaczyna od podanego adresu i czyta do pierwszego NUL. Przy każdym wydruku przez `%s` terminator musi już istnieć w granicach dostępnej pamięci.

| Stan tablicy | `[0]` | `[1]` | `[2]` | `[3]` | Tekst dla `%s` |
|---|---|---|---|---|---|
| Początek | H | i | **`'\0'`** | niezainicjalizowany | Hi |
| Przygotuj koniec | H | i | **`'\0'`** | `'\0'` | Hi |
| Dopisz znak | H | i | ! | **`'\0'`** | Hi! |
| Wstaw NUL w środku | H | **`'\0'`** | ! | `'\0'` | H |
| Wstaw NUL na początku | **`'\0'`** | `'\0'` | ! | `'\0'` | pusty |

Pogrubienie wskazuje **pierwszy** terminator. Przed zastąpieniem starego terminatora wpiszemy nowy.

**Przewidź:** kiedy wykrzyknik pojawi się w wydruku? Czy zniknie z tablicy, gdy wcześniej wstawimy NUL?

In [ ]:
#include <stdio.h>

int main(void)
{
    char napis[4];
    napis[0] = 'H';
    napis[1] = 'i';
    napis[2] = '\0';

    printf("Znaki: %c %c\n", napis[0], napis[1]);
    printf("[%s]\n", napis);

    napis[3] = '\0';
    napis[2] = '!';
    printf("[%s]\n", napis);

    napis[1] = '\0';
    printf("[%s], pod indeksem 2 jest '%c'\n", napis, napis[2]);

    napis[0] = '\0';
    printf("[%s]\n", napis);
    return 0;
}

**Interpretacja.** Kolejne napisy to `Hi`, `Hi!`, `H` i napis pusty. Wstawienie NUL zmieniło granicę odczytu przez `%s`. Pod indeksem 2 pozostał wykrzyknik. Na początku miejsce pod indeksem 3 nie miało ustalonej wartości. Pierwszy wydruk sięgał tylko do indeksu 2.

**Spróbuj.** W ostatnim stanie ustaw `napis[0] = 'A'`. Przewidź wynik `printf("[%s]\n", napis)` i wskaż terminator.

## W11.3 — Długość, miejsce na NUL i pojemność

Ile miejsc potrzeba na tekst i ile jest ich w tablicy?

Listę znaków można wpisać w klamrach i zakończyć NUL. Literał w cudzysłowie już zawiera końcowy NUL. Gdy pomijamy rozmiar, kompilator wywnioskuje go z inicjalizatora.

| Deklaracja | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| `char duzy[6] = "kot";` | k | o | t | **`'\0'`** | `'\0'` | `'\0'` |

`strlen(duzy)` podaje **długość** napisu w bajtach przed pierwszym NUL. `strlen(duzy) + 1` to **wymagana liczba bajtów** na kopię całego napisu. `sizeof duzy` to **pojemność** tej tablicy w bajtach. Zainicjalizowanie tablicy krótszym literałem zeruje niewykorzystane elementy. Przy ręcznych przypisaniach, jak w W11.2, pozycje, do których nic nie wpisano, mogą pozostać niezainicjalizowane.

**Przewidź:** w której tablicy zmieści się jeszcze jeden znak i terminator?

In [ ]:
#include <stdio.h>
#include <string.h>

int main(void)
{
    char lista[] = {'k', 'o', 't', '\0'};
    char maly[] = "kot";
    char duzy[6] = "kot";

    printf("lista: %s, sizeof = %zu\n", lista, sizeof lista);
    printf("maly: %s, sizeof = %zu\n", maly, sizeof maly);
    printf("duzy: długość = %zu, potrzeba = %zu, pojemność = %zu\n",
           strlen(duzy), strlen(duzy) + 1, sizeof duzy);
    printf("duzy[3..5]: %d %d %d\n", duzy[3], duzy[4], duzy[5]);
    return 0;
}

**Interpretacja.** `lista` i `maly` mają po cztery miejsca, wszystkie zajęte przez trzy znaki i terminator. `duzy` ma długość napisu 3, wymaganą liczbę bajtów 4 i pojemność 6. Za pierwszym terminatorem są dwa zera.

**Spróbuj.** Zmień literał w deklaracji `duzy` na `"kotek"`. Przewidź nowe wartości. W `char a[3] = "kot";` mieszczą się trzy litery. Przed użyciem `%s` trzeba zapewnić terminator.

## W11.4 — Dwa koty, dwa sposoby przechowywania

Co się zmieni, kiedy wpiszemy nowy znak, a co po zmianie adresu?

Czarnego kota zapiszemy w tablicy `a`. Białego opisuje literał, którego adres przechowamy w `p`. Wskaźnik `q` dostanie adres pierwszego elementu tablicy `a`.

    q ──► a: [C][z][a][r][n][y][ ][k][o][t]['\0']
    p ──►    [B][i][a][l][y][ ][k][o][t]['\0']  literał

`a` jest tablicą. W inicjalizacji `q = a` wyrażenie `a` daje adres pierwszego elementu. `q` jest osobnym obiektem, który przechowuje ten adres. Polecenie `a = "Kot";` jest niedopuszczalne, bo C nie pozwala przypisać wartości całej tablicy. Jej pojedyncze elementy można zmieniać.

`const char *p` pozwala odczytywać znaki przez `p` oraz przypisywać mu inny adres. Próba zapisu do literału ma niezdefiniowane zachowanie, również po uzyskaniu adresu o typie `char *`. Wskaźnik `char *q` w tym przykładzie prowadzi do modyfikowalnej tablicy. Pojemność znamy z `sizeof a` w miejscu deklaracji tej tablicy. `sizeof q` podaje rozmiar wskaźnika, także gdy występuje on jako parametr funkcji.

**Przewidź:** po zapisie przez `q` i zmianie wartości `p` ile napisów zmieni treść?

In [ ]:
#include <stdio.h>

int main(void)
{
    char a[] = "Czarny kot";
    const char *p = "Bialy kot";
    char *q = a;

    printf("a: %s, p: %s\n", a, p);
    q[0] = 'c';
    printf("Po zapisie przez q: %s\n", a);

    p = a;
    printf("p: %s, p i q wskazują to samo: %d\n", p, p == q);
    return 0;
}

**Interpretacja.** Zapis przez `q` zamienia `C` na `c` w tablicy. Przypisanie `p = a` kopiuje adres. Oba wskaźniki czytają teraz ten sam napis, a literał z białym kotem zachowuje treść.

**Spróbuj.** Po `p = a` zmień ostatnią literę przez `q` i wypisz napis przez `p`. Co potwierdza ten wydruk?

## W11.5 — Słowo, linia i plik

Czy dwa słowa oddzielone spacją da się wczytać za jednym razem?

### Wczytanie jednego słowa

Dla bufora `char tekst[100]` wywołanie `scanf("%99s", tekst)` pomija początkowe białe znaki i wczytuje do 99 bajtów słowa. Zostaje miejsce na NUL. Wynik równy 1 oznacza, że można odczytać napis. Gdy funkcja zatrzyma się na spacji lub osiągnie limit szerokości, dalsze dane czekają w strumieniu.

**Przewidź.** Który fragment wejścia `Ala ma kota\n` trafi do tablicy? Jaki bajt pobierze potem `fgetc`?

W notebooku program czyta przygotowane dane z pliku tymczasowego. `fscanf` przyjmuje jawnie strumień. W wariancie terminalowym `scanf` czyta ze `stdin`. Zapisz komórkę jako `slowo.c` i uruchom poniższe polecenia, aby podać własne wejście.

```sh
gcc -std=c11 -DW11_STDIN slowo.c -o slowo
printf 'Ala ma kota\n' | ./slowo
```

In [ ]:
#include <stdio.h>

int main(void)
{
    char tekst[100];

#ifdef W11_STDIN
    FILE *wejscie = stdin;
    int wynik = scanf("%99s", tekst);
#else
    FILE *wejscie = tmpfile();
    if (wejscie == NULL) {
        perror("Plik tymczasowy");
        return 1;
    }
    if (fputs("Ala ma kota\n", wejscie) == EOF ||
        fseek(wejscie, 0, SEEK_SET) != 0) {
        perror("Przygotowanie danych");
        fclose(wejscie);
        return 1;
    }
    int wynik = fscanf(wejscie, "%99s", tekst);
#endif

    if (wynik == 1) {
        printf("Słowo: [%s]\n", tekst);
        int bajt = fgetc(wejscie);
        if (bajt == EOF) {
            if (!ferror(wejscie))
                printf("Brak następnego bajtu.\n");
        } else if (bajt == '\n') {
            printf("Następny bajt: znak nowej linii.\n");
        } else {
            printf("Następny bajt: [%c]\n", bajt);
        }
    } else if (!ferror(wejscie)) {
        printf("Brak słowa na wejściu.\n");
    }

    int blad = ferror(wejscie);
    if (blad)
        fprintf(stderr, "Błąd odczytu.\n");
    if (wejscie != stdin && fclose(wejscie) == EOF) {
        perror("Zamykanie pliku");
        return 1;
    }
    return wynik != 1 || blad;
}

**Interpretacja.** W tablicy jest `Ala`. Następnym bajtem wejścia jest spacja. Po jej pobraniu w strumieniu pozostaje `ma kota\n`. Jeśli później użyjesz `fgets` na tym samym strumieniu, odczyt ruszy od bieżącej pozycji.

**Spróbuj.** Dodaj spacje przed `Ala`, a po słowie wstaw tabulator. Przewidź zawartość tablicy i następny pobrany bajt.

### Wczytanie linii

`fgets` pobiera także spacje. Przy pojemności `n` czyta najwyżej `n - 1` bajtów, zapisuje końcowy NUL i po udanym odczycie zwraca adres bufora. Jeśli w odczytanym fragmencie zmieścił się znak `\n`, zostaje on w tablicy. Wynik `NULL` oznacza koniec danych przed odczytem choćby jednego bajtu albo błąd. `ferror` pozwala sprawdzić błąd strumienia.

```c
char *fgets(char *restrict s, int n, FILE *restrict stream);
```

Przyjmijmy plik tekstowy bez bajtów NUL. W poniższym programie bufor ma dziewięć miejsc. Przyjmujemy najwyżej siedem bajtów treści linii. Dwa pozostałe miejsca pozwalają zmieścić `\n` i NUL. Gdy `fgets` da osiem bajtów bez `\n`, linia przekracza limit: doczytamy i pominiemy jej resztę. Krótki odczyt bez `\n` jest ostatnią linią zakończoną przez EOF. W kodzie `strchr(linia, '\n')` zwraca adres znaku nowej linii albo `NULL`, gdy tego znaku w buforze nie ma. Zmieniamy znaleziony element na NUL.

**Przewidź.** W pliku są kolejno `Ala ma`, pusta linia, `1234567`, `12345678ale` i `kot` bez końcowego `\n`. Które linie program wypisze? Jaki numer dostanie `kot`?

Notebook tworzy plik przez `tmpfile`. C11 usuwa taki plik przy zamknięciu. Ten sam program można sprawdzić w terminalu na standardowym wejściu. Zapisz komórkę jako `linie.c`.

```sh
gcc -std=c11 -DW11_STDIN linie.c -o linie
printf 'Ala ma\n\n1234567\n12345678ale\nkot' | ./linie
```

In [ ]:
#include <stdio.h>
#include <string.h>

int main(void)
{
#ifdef W11_STDIN
    FILE *wejscie = stdin;
#else
    FILE *wejscie = tmpfile();
    if (wejscie == NULL) {
        perror("Plik tymczasowy");
        return 1;
    }
    if (fputs("Ala ma\n\n1234567\n12345678ale\nkot", wejscie) == EOF ||
        fseek(wejscie, 0, SEEK_SET) != 0) {
        perror("Przygotowanie danych");
        fclose(wejscie);
        return 1;
    }
#endif

    char linia[9];
    size_t numer = 0;
    while (fgets(linia, sizeof linia, wejscie) != NULL) {
        ++numer;
        char *nowa_linia = strchr(linia, '\n');
        if (nowa_linia != NULL) {
            *nowa_linia = '\0';
        } else if (strlen(linia) == sizeof linia - 1) {
            int bajt;
            do {
                bajt = fgetc(wejscie);
            } while (bajt != '\n' && bajt != EOF);
            printf("%zu: pominięto zbyt długą linię\n", numer);
            if (bajt == EOF)
                break;
            continue;
        }
        printf("%zu: [%s]\n", numer, linia);
    }

    int blad = ferror(wejscie);
    if (blad)
        fprintf(stderr, "Błąd odczytu.\n");
    if (wejscie != stdin && fclose(wejscie) == EOF) {
        perror("Zamykanie pliku");
        return 1;
    }
    return blad != 0;
}

**Interpretacja.** Spacja w `Ala ma` pozostała w tekście. Pusta linia dała `[]`, ponieważ po usunięciu `\n` pierwszy element to NUL. Siedem cyfr wypełniło dopuszczalną długość, a czwarta linia przekroczyła limit. Po pominięciu jej reszty `kot` zachował numer 5 i został wypisany mimo braku końcowego `\n`.

**Spróbuj.** Najpierw usuń znak `\n` po linii z siedmioma cyframi i zakończ w tym miejscu dane wejściowe. Powtórz próbę z ośmioma cyframi. Przed uruchomieniem określ, ile bajtów odczyta każde `fgets`.

## W11.6 — Pętle, które czytają napisy

Jak ustalić długość napisu i porównać go z innym?

Długość to liczba kolejnych bajtów przed NUL. Zaczynamy od indeksu 0 i zwiększamy go dopiero po stwierdzeniu, że bieżący element należy do treści.

Do porównania idziemy po obu napisach równocześnie. Pierwsze różne bajty rozstrzygają kolejność. Jeśli dojdziemy do NUL w obu napisach naraz, są równe. Dla prefiksu, takiego jak `"kot"` w `"kotek"`, porównanie rozstrzyga NUL krótszego napisu i następny bajt dłuższego. Bajty w porównaniu interpretujemy jako `unsigned char`, aby uzyskać taką samą kolejność jak w `strcmp`. To porządek według bajtów. Reguły słownika zależą od języka.

Parametry `const char *` pozwalają funkcji odczytać napis. Obie funkcje potrzebują poprawnie zakończonych napisów.

**Przewidź.** Podaj dwie długości i relację dla każdej pary z programu. Zaznacz indeks pierwszej różnicy.

In [ ]:
#include <stdio.h>

static size_t mystrlen(const char *s)
{
    size_t i = 0;
    while (s[i] != '\0')
        ++i;
    return i;
}

static int mystrcmp(const char *s1, const char *s2)
{
    size_t i = 0;
    while (s1[i] == s2[i] && s1[i] != '\0')
        ++i;

    unsigned char a = (unsigned char)s1[i];
    unsigned char b = (unsigned char)s2[i];
    if (a < b)
        return -1;
    if (a > b)
        return 1;
    return 0;
}

static void pokaz(const char *a, const char *b)
{
    int wynik = mystrcmp(a, b);
    char relacja = '=';
    if (wynik < 0)
        relacja = '<';
    if (wynik > 0)
        relacja = '>';
    printf("[%s] (%zu) %c [%s] (%zu)\n",
           a, mystrlen(a), relacja, b, mystrlen(b));
}

int main(void)
{
    pokaz("", "");
    pokaz("", "k");
    pokaz("kot", "kot");
    pokaz("kot", "kotek");
    pokaz("pies", "kot");
    return 0;
}

**Interpretacja.** Dwa puste napisy są równe, podobnie jak dwa napisy `kot`. Pusty napis poprzedza `k`, a `kot` poprzedza `kotek`. W tych dwóch parach rozstrzyga terminator. W parze `pies`, `kot` wystarczy porównać `p` z `k`. Czytamy znak relacji z warunków `wynik < 0`, `wynik == 0` i `wynik > 0`.

**Spróbuj.** Zamień miejscami argumenty dwóch ostatnich wywołań. Czy indeks pierwszej różnicy się zmieni?

## W11.7 — Sześć operacji z <string.h>

Jak skopiować, połączyć i przeszukać napisy?

### Długość, porównanie i przepisanie bajtów

Deklaracje pochodzą z nagłówka `<string.h>`.

```c
size_t strlen(const char *s);
int strcmp(const char *s1, const char *s2);
char *strcpy(char *restrict dest, const char *restrict src);
char *strcat(char *restrict dest, const char *restrict src);
```

| Funkcja | Co robi | O czym trzeba pamiętać |
|---|---|---|
| `strlen(s)` | Liczy bajty przed pierwszym NUL. | Wynik ma typ `size_t` i opisuje długość napisu. Pojemność trzeba znać osobno. |
| `strcmp(a, b)` | Porównuje napisy według bajtów interpretowanych jako `unsigned char`. | Testuj znak wyniku. Biblioteka może zwrócić inną wartość ujemną lub dodatnią niż −1 lub 1. |
| `strcpy(cel, src)` | Kopiuje cały napis razem z NUL i zwraca `cel`. | Potrzebuje co najmniej `strlen(src) + 1` bajtów w celu. |
| `strcat(cel, src)` | W miejscu NUL w celu zaczyna kopiować `src` razem z jego NUL i zwraca `cel`. | W celu musi już być poprawny napis. Pojemność musi pomieścić obie treści i nowy terminator. |

Wszystkie źródła muszą być poprawnie zakończone. Przy kopiowaniu i dopisywaniu obszary źródła i celu muszą się mieścić w swoich granicach i być rozłączne.

**Przewidź.** Jakie są kolejne długości napisu w `cel`? Czy po dopisaniu zostanie miejsce na wykrzyknik?

In [ ]:
#include <stdio.h>
#include <string.h>

int main(void)
{
    char cel[6];

    strcpy(cel, "ko");
    printf("[%s], długość: %zu\n", cel, strlen(cel));

    strcat(cel, "tek");
    printf("[%s], długość: %zu, pojemność: %zu\n", cel, strlen(cel), sizeof cel);

    printf("Czy [kot] poprzedza [%s]? %d\n", cel, strcmp("kot", cel) < 0);
    return 0;
}

**Interpretacja.** Po skopiowaniu mamy `ko` o długości 2. Dopisanie od indeksu 2 daje `kotek` o długości 5. Terminator trafia na indeks 5, więc bufor jest pełny. `strcmp("kot", cel) < 0` jest prawdą.

### Gdzie jest pierwsze trafienie?

```c
char *strchr(const char *s, int c);
char *strstr(const char *s1, const char *s2);
```

`strchr(s, c)` szuka pierwszego znaku `c` po jego konwersji na `char`. Można też szukać samego NUL. `strstr(s, wzorzec)` szuka pierwszego wystąpienia całego wzorca. Pusty wzorzec pasuje od początku. Przy braku trafienia obie funkcje zwracają `NULL`.

    indeks: 0   1   2   3   4   5   6   7   8   9  10   11
    s:     [A] [l] [a] [ ] [m] [a] [ ] [k] [o] [t] [a] ['\0']
                                        ↑
                                        p = strstr(s, "kota")

Po sukcesie `p` wskazuje element tej samej tablicy. Różnica `p - s` daje indeks 7, a wydruk od `p` daje sufiks `kota`. Najpierw sprawdzamy `p`, później odejmujemy wskaźniki lub czytamy dane. `%td` wypisuje różnicę wskaźników.

**Przewidź.** Które trafienie wypada na początku? Od którego miejsca wydrukuje tekst `strchr(s, 'm')`? Co zwróci wyszukanie `"psa"`?

In [ ]:
#include <stdio.h>
#include <string.h>

static void pokaz_trafienie(const char *s, const char *p)
{
    if (p == NULL) {
        printf("Brak trafienia.\n");
        return;
    }
    printf("Indeks: %td, sufiks: [%s]\n", p - s, p);
}

int main(void)
{
    const char s[] = "Ala ma kota";

    pokaz_trafienie(s, strchr(s, 'm'));
    pokaz_trafienie(s, strchr(s, 'z'));
    pokaz_trafienie(s, strstr(s, "Ala"));
    pokaz_trafienie(s, strstr(s, "kota"));
    pokaz_trafienie(s, strstr(s, "psa"));
    return 0;
}

**Interpretacja.** `m` daje indeks 4 i sufiks `ma kota`, a `Ala` zaczyna się pod indeksem 0. `kota` zaczyna się pod indeksem 7. Dla `z` i `psa` wypisujemy tylko komunikat o braku trafienia.

**Spróbuj.** Wyszukaj `'a'`, potem `'\0'`. Wskaż na schemacie oba miejsca i przewidź sufiksy. Ile bajtów musiałby mieć bufor, gdybyś chciał skopiować sufiks `ma kota`?

## W11.8 — Dwa wejścia i jeden nowy napis

Jak dobrać rozmiar przydzielanej pamięci do danych wejściowych?

Podamy maksymalną długość obu napisów w pierwszej linii. W tym doświadczeniu limit jest jedną cyfrą od 1 do 9. Dwie kolejne linie zawierają napisy. Liczymy bajty, zachowujemy spacje i dopuszczamy puste linie. Dane są tekstem bez wewnętrznego NUL.

Dla odczytu rezerwujemy po `limit + 2` bajty na treść, znak nowej linii i NUL. Po odczycie usuwamy `\n`, jeśli wystąpił. Wynik wymaga dokładnie `strlen(a) + strlen(b) + 1` bajtów. Przy dwóch napisach po najwyżej 9 bajtów suma wynosi najwyżej 19 i mieści się w `size_t`.

    a: [ A ][ l ][ a ][ NUL ]
    b: [ ␣ ][ m ][ a ][ ␣ ][ k ][ o ][ t ][ a ][ NUL ]

    wynik: [ A ][ l ][ a ][ ␣ ][ m ][ a ][ ␣ ][ k ][ o ][ t ][ a ][ NUL ]

    strlen(a) + strlen(b) + 1 = 3 + 8 + 1 = 12 bajtów

Najpierw `strcpy` tworzy w buforze poprawny napis, a potem `strcat` dopisuje drugi od miejsca zajętego przez pierwszy NUL.

**Przewidź.** Jaki będzie wynik dla powyższych danych i ile bajtów zajmie sam tekst? Gdzie znajdzie się NUL po skopiowaniu pierwszego napisu, a gdzie po dopisaniu drugiego?

Komórka korzysta z przygotowanych danych. Aby czytać ze standardowego wejścia, zapisz ją jako `polacz.c` i skompiluj wariant terminalowy.

```sh
gcc -std=c11 -DW11_STDIN polacz.c -o polacz
printf '9\nAla\n ma kota\n' | ./polacz
```

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static int wczytaj(FILE *wejscie, char *tekst, int limit)
{
    if (fgets(tekst, limit + 2, wejscie) == NULL) {
        fprintf(stderr, "%s\n", ferror(wejscie) ? "Błąd odczytu." : "Brak linii z napisem.");
        return 0;
    }
    char *koniec = strchr(tekst, '\n');
    if (koniec != NULL)
        *koniec = '\0';
    if (strlen(tekst) > (size_t)limit) {
        fprintf(stderr, "Napis przekracza limit.\n");
        return 0;
    }
    return 1;
}

static int polacz(FILE *wejscie)
{
    char rozmiar[3];
    if (fgets(rozmiar, sizeof rozmiar, wejscie) == NULL) {
        fprintf(stderr, "%s\n", ferror(wejscie) ? "Błąd odczytu." : "Brak limitu.");
        return 1;
    }
    if (rozmiar[0] < '1' || rozmiar[0] > '9' || rozmiar[1] != '\n') {
        fprintf(stderr, "Limit musi być cyfrą od 1 do 9.\n");
        return 1;
    }
    int limit = rozmiar[0] - '0';
    char *a = malloc((size_t)limit + 2);
    char *b = malloc((size_t)limit + 2);
    char *wynik = NULL;
    int status = 1;

    if (a == NULL || b == NULL) {
        fprintf(stderr, "Brak pamięci na dane wejściowe.\n");
    } else if (wczytaj(wejscie, a, limit) && wczytaj(wejscie, b, limit)) {
        size_t pojemnosc = strlen(a) + strlen(b) + 1;
        wynik = malloc(pojemnosc);
        if (wynik == NULL) {
            fprintf(stderr, "Brak pamięci na wynik.\n");
        } else {
            strcpy(wynik, a);
            strcat(wynik, b);
            printf("Wynik: [%s], rozmiar z NUL w bajtach: %zu\n", wynik, pojemnosc);
            status = 0;
        }
    }

    free(wynik);
    free(b);
    free(a);
    return status;
}

int main(void)
{
#ifdef W11_STDIN
    return polacz(stdin);
#else
    FILE *wejscie = tmpfile();
    if (wejscie == NULL) {
        perror("Plik tymczasowy");
        return 1;
    }
    if (fputs("9\nAla\n ma kota\n", wejscie) == EOF ||
        fseek(wejscie, 0, SEEK_SET) != 0) {
        perror("Przygotowanie danych");
        fclose(wejscie);
        return 1;
    }
    int status = polacz(wejscie);
    if (fclose(wejscie) == EOF) {
        perror("Zamykanie pliku");
        return 1;
    }
    return status;
#endif
}

**Interpretacja.** Wynik to `Ala ma kota`. Po `strcpy` terminator jest na pozycji 3. `strcat` zapisuje od tej pozycji osiem bajtów, po czym umieszcza NUL na pozycji 11. Każdy przydzielony obszar zwalniamy raz. Jeśli wskaźnik ma wartość `NULL`, `free` nie wykonuje żadnej operacji.

**Spróbuj.** Użyj danych `1\n\n\n`. Przewidź zawartość i rozmiar trzeciego napisu. Następnie podaj `1\nA\nBC\n` i wskaż, w którym miejscu program odrzuci dane.

**Przeanalizuj bez uruchamiania.** Załóż, że oba źródła są poprawnymi napisami i że `malloc` się powiódł. Jakie dwa niezależne błędy zawiera poniższy zapis? Co pozostanie do naprawienia, jeśli zwiększysz bufor o jeden bajt?

```c
char *wynik = malloc(strlen(a) + strlen(b));
strcat(wynik, a);
strcat(wynik, b);
```

Zapis pomija miejsce na NUL. Ponadto pierwszy `strcat` szuka terminatora w niezainicjalizowanym obszarze przydzielonym przez `malloc`. Samo powiększenie obszaru nie tworzy w nim napisu.

## W11.9 — Które linie pasują?

Jak policzyć linie pliku, w których występuje szukany fragment?

### Prosty filtr pliku tekstowego

Wczytamy **niepusty fragment**, potem otworzymy plik i sprawdzimy każdą jego linię przez `strstr`. Wypiszemy numer i tekst każdej pasującej linii. Nawet jeśli szukany fragment pojawia się w linii kilka razy, ta linia zwiększa licznik tylko raz.

Przyjmujemy tekst bez bajtów NUL. Fragment może mieć najwyżej 40 bajtów, a linia pliku najwyżej 80 bajtów treści. W obu buforach dodajemy miejsca na `\n` i NUL. Jak w W11.5, zbyt długą linię doczytujemy do `\n` lub EOF i pomijamy ją w wyszukiwaniu. Następna linia otrzymuje kolejny numer. Krótka ostatnia linia może skończyć się bez `\n`.

**Przewidź.** Szukamy `kot`. Plik zawiera `Pies śpi.`, `Ala ma kota.`, pustą linię i `To kot i drugi kot.` bez końcowego `\n`. Które numery zostaną wypisane? Jaki będzie licznik?

W notebooku przygotujemy dwa małe pliki tymczasowe: jeden z fragmentem, drugi z tekstem do przeszukania. `tmpfile` otwiera plik, który znika po zamknięciu. W terminalu wariant `W11_STDIN` czyta fragment ze standardowego wejścia i otwiera w trybie odczytu plik `tekst.txt`. Przygotuj taki plik z podanymi wyżej liniami w katalogu programu i zapisz komórkę jako `szukaj.c`.

```sh
gcc -std=c11 -DW11_STDIN szukaj.c -o szukaj
printf 'kot\n' | ./szukaj
```

In [ ]:
#include <stdio.h>
#include <string.h>

enum { LIMIT_FRAGMENTU = 40, LIMIT_LINII = 80 };

static int wczytaj_fragment(FILE *wejscie, char fragment[LIMIT_FRAGMENTU + 2])
{
    if (fgets(fragment, LIMIT_FRAGMENTU + 2, wejscie) == NULL) {
        fprintf(stderr, "%s\n", ferror(wejscie) ? "Błąd odczytu fragmentu." :
                                                  "Brak fragmentu na wejściu.");
        return 1;
    }

    char *koniec = strchr(fragment, '\n');
    if (koniec != NULL)
        *koniec = '\0';
    if (fragment[0] == '\0' || strlen(fragment) > LIMIT_FRAGMENTU) {
        fprintf(stderr, "Fragment musi mieć od 1 do %d bajtów.\n", LIMIT_FRAGMENTU);
        return 1;
    }
    return 0;
}

static int szukaj(FILE *plik, const char *fragment)
{
    char linia[LIMIT_LINII + 2];
    size_t numer = 0;
    size_t trafienia = 0;

    while (fgets(linia, sizeof linia, plik) != NULL) {
        ++numer;
        char *koniec = strchr(linia, '\n');
        if (koniec != NULL) {
            *koniec = '\0';
        } else if (strlen(linia) > LIMIT_LINII) {
            int bajt;
            do {
                bajt = fgetc(plik);
            } while (bajt != '\n' && bajt != EOF);
            printf("Pominięto linię %zu: ponad %d bajtów.\n", numer, LIMIT_LINII);
            if (bajt == EOF)
                break;
            continue;
        }

        if (strstr(linia, fragment) != NULL) {
            printf("%zu: %s\n", numer, linia);
            ++trafienia;
        }
    }

    if (ferror(plik)) {
        fprintf(stderr, "Błąd odczytu pliku.\n");
        return 1;
    }
    printf("Pasujące linie: %zu\n", trafienia);
    return 0;
}

#ifndef W11_STDIN
static FILE *plik_z_tekstem(const char *tekst)
{
    FILE *plik = tmpfile();
    if (plik == NULL)
        return NULL;
    if (fputs(tekst, plik) == EOF || fseek(plik, 0, SEEK_SET) != 0) {
        fclose(plik);
        return NULL;
    }
    return plik;
}
#endif

int main(void)
{
    char fragment[LIMIT_FRAGMENTU + 2];

#ifdef W11_STDIN
    if (wczytaj_fragment(stdin, fragment) != 0)
        return 1;
    FILE *plik = fopen("tekst.txt", "r");
#else
    FILE *wejscie = plik_z_tekstem("kot\n");
    if (wejscie == NULL) {
        fprintf(stderr, "Nie udało się przygotować fragmentu.\n");
        return 1;
    }
    int wynik = wczytaj_fragment(wejscie, fragment);
    if (fclose(wejscie) == EOF) {
        fprintf(stderr, "Błąd zamknięcia wejścia.\n");
        return 1;
    }
    if (wynik != 0)
        return 1;
    FILE *plik = plik_z_tekstem("Pies śpi.\nAla ma kota.\n\nTo kot i drugi kot.");
#endif

    if (plik == NULL) {
        fprintf(stderr, "Nie udało się otworzyć pliku z tekstem.\n");
        return 1;
    }
    int status = szukaj(plik, fragment);
    if (fclose(plik) == EOF) {
        fprintf(stderr, "Błąd zamknięcia pliku.\n");
        return 1;
    }
    return status;
}

**Interpretacja.** Pasują linie 2 i 4. Licznik daje 2, choć w linii 4 fragment występuje dwukrotnie. Pusta linia zajmuje numer 3. Brak ostatniego `\n` nie zmienia wyniku.

### Sprawdź, co rozumiesz

1. Co wypisze `printf("[%s]", s)` dla `char s[] = {'a', '\0', 'b', '\0'};`? Ile wynoszą `strlen(s)` i `sizeof s`?
2. Ile miejsc potrzebujesz na połączenie napisów o długościach 4 i 5? Pod jakim indeksem znajdzie się NUL?
3. Jaki znak ma wynik `strcmp("k", "kot")`? Na których bajtach rozstrzyga się porównanie?
4. Kiedy wolno obliczyć `p - s` po wywołaniu `strchr(s, 'a')`? Co oznacza wynik różnicy?

### Zmień jeden szczegół

- W filtrze wyszukaj fragment `ryba`, a potem `Ala ma`. Podaj oczekiwane liczniki przed uruchomieniem.
- Dopisz w pliku linię o długości 81 bajtów, a bezpośrednio po niej krótką linię z `kot`. Przewidź komunikat i numer trafienia.
- W programie z W11.8 połącz dwa napisy ze spacją w środku. Najpierw ustal nowy rozmiar bufora i wskaż miejsce na końcowy NUL.